# Cohere Rerank
Cohere(캐나다)는 기업용 AI, 자연어처리, 검색 및 생성형 AI 분야에서 강력한 솔루션을 제공하는 플랫폼이다.

In [1]:
%pip install cohere

INFO: pip is looking at multiple versions of pydantic to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ------------------------------------ --- 1.8/2.0 MB 16.7 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 12.5 MB/s  0:00:00

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.46.3

    Uninstalling pydantic_core-2.46.3:

      Successfully uninstalled pydantic_core-2.46.3

   -------- ------------------------------- 1/5 [pydantic-core]
   -------- ------------------------------- 1/5 [pydantic-core]
   -------- ------------------------------- 1/5 [pydantic-core]
   -------- ------------------------------- 1/5 [pydantic-core]
   -------- ------------------------------- 1/5 [pydantic-core]
   -------- ------------------------------- 1/5 [pydantic-core]
   -------- ------------------------------- 1/5 [pydantic-core]

  You can safely remove it manually.

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 환경설정

In [1]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

## 데이터 로드

In [2]:
import pandas as pd

document_df = pd.read_csv("data/documents.csv")
queries_df = pd.read_csv("data/queries.csv")
queries_df

,query_id,query_text,relevant_doc_ids
0,Q1,제주도 올레길 트레킹 코스 추천,D1=2
1,Q2,전주 비빔밥 vs 진주 비빔밥 재료 및 맛 차이,D2=3
2,Q3,걸스데이 대표 히트곡 목록 알려줘,D3=3
3,Q4,훈민정음 창제 배경과 세종대왕의 의의,D4=3
4,Q5,이순신 장군이 명량 해전에서 사용한 전술은 무엇인가?,D5=3
5,Q6,2024년 기후 변화 주요 지표와 한국의 탄소 중립 정책,D6=3
6,Q7,한국 AI 윤리 이슈와 관련 정책 사례는?,D7=3;D25=2
7,Q8,서울 지하철 환승 시 T-money 사용 방법,D8=2
8,Q9,판소리 춘향가 줄거리와 공연 특징,D9=3
9,Q10,한국 축구 대표팀 2002년 한일 월드컵 4강 진출 이유,D10=2


## 검색기 준비

In [3]:
from konlpy.tag import Okt
from rank_bm25 import BM25Okapi

okt = Okt()
tokenized_docs = [okt.morphs(content) for content in document_df['content']]
bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query, top_k=5):
    """
    BM25로 질문과 관련 있는 상위 문서 ID를 반환한다.
    """
    query_token = okt.morphs(query)
    scores = bm25.get_scores(query_token)
    sorted_idx = sorted(range(len(scores)), key=lambda i:scores[i], reverse=True)
    ranked_docs = [document_df['doc_id'].iloc[i] for i in sorted_idx[:top_k]]
    return ranked_docs

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings
)

def dense_search(query,top_k=5):
    """Dense retrieval로 질문과 관련있는 상위 문서 ID를 반환한다."""
    docs = vector_store.similarity_search(query,k=top_k)
    return [doc.metadata["doc_id"] for doc in docs]

## Cohere Reranking

**Bi-Encoder와 Cross-Encoder 비교**

**Bi-Encoder**와 **Cross-Encoder**는 문장 쌍의 관계(예: 유사도, 연관성 등)를 계산할 때 사용하는 대표적인 두 가지 구조이다. 각각의 구조와 특징을 간단하게 비교하면 다음과 같다.

| 구조         | 입력 방식                                      | 연산 속도         | 성능(정확도)         | 특징 요약                        |
|:------------:|:---------------------------------------------:|:----------------:|:--------------------:|:-------------------------------:|
| **Bi-Encoder**   | 두 문장을 각각 독립적으로 임베딩                | 빠름              | 다소 낮음             | 임베딩 미리 계산/저장 가능, 대량 비교 적합 |
| **Cross-Encoder**| 두 문장을 [SEP]으로 연결해 한 번에 입력           | 느림              | 높음                  | 문장 간 상호작용 정보 최대 활용, 소규모 비교 적합 |

**설명**

- **Bi-Encoder**  
  - 두 문장을 각각 독립적으로 인코더(BERT 등)에 넣어 임베딩 벡터를 만든다.
  - 만들어진 임베딩 벡터끼리 코사인 유사도 등으로 비교한다.
  - 임베딩을 미리 계산해 둘 수 있으므로, 대규모 문장 비교에서 매우 빠른 속도를 낼 수 있다.
  - 하지만 문장 간의 미세한 상호작용 정보가 손실될 수 있어, Cross-Encoder에 비해 정확도가 낮다.

- **Cross-Encoder**  
  - 두 문장을 [SEP] 토큰으로 연결해 한 번에 인코더에 넣는다.
  - 모델이 두 문장 사이의 상호작용 정보를 직접 활용해 결과(유사도 등)를 바로 출력한다.
  - 모든 문장 쌍마다 모델 연산이 필요하므로, 비교해야 할 문장이 많아질수록 속도가 매우 느려진다.
  - 하지만 문장 간 관계를 더욱 정확하게 파악할 수 있어, 성능(정확도)이 높다.

**정리**
- **Bi-Encoder**는 속도가 빠르지만, 성능(정확도)은 Cross-Encoder보다 낮다.
- **Cross-Encoder**는 성능이 뛰어나지만, 연산량이 많아 속도가 느리다.
- 실제로는 Bi-Encoder로 후보군을 먼저 빠르게 좁히고, Cross-Encoder로 최종 순위를 정하는 식으로 두 구조를 조합해 사용하는 경우가 많다.

## 후보 문서 집합 생성

In [5]:
bm25_candidates = {}
for dix, row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']
    bm25_candidates[qid] = bm25_search(query_text,top_k=20)

dense_candidates = {}
for dix, row in queries_df.iterrows():
    qid = row['query_id']
    query_text = row['query_text']
    dense_candidates[qid] = dense_search(query_text,top_k=20)

In [14]:
# Cohere Rerank API 호출 함수
# 후보 문서를 query와 다시 비교하여 관련도 순으로 재정렬한다.
import cohere
from cohere import TooManyRequestsError
import time

# COHERE_API_KEY 환경변수가 설정 되어 있어야 함
co = cohere.Client()

# 한국어 문서를 다루므로 multilinggual rerank 모델 사용
COHERE_RERANK_MODEL = 'rerank-multilingual-v3.0'

def cohere_rerank(query,candidates,max_retries=1,wait_seconds=10):
    # candidates에는 doc_id만 담겨있으므로 실제 문서 내용을 조회해서 넘겨준다.
    texts = [
        document_df.loc[document_df['doc_id'] == doc_id, 'content'].values[0]
        for doc_id in candidates
    ]

    def call_rerank_api():

        response = co.rerank(
            model=COHERE_RERANK_MODEL,
            query=query,
            documents=texts
        )

        reranked = sorted(response.results, key=lambda x : x.relevance_score, reverse=True)

        return [candidates[r.index] for r  in reranked]
    
    for attempt in range(max_retries + 1):
        try:
            return call_rerank_api()
        except TooManyRequestsError:
            if attempt < max_retries:
                print(f'TooManyRequestsError : {wait_seconds}초 후 재시도 합니다. (시도 {attempt + 1}/{max_retries})')
                time.sleep(wait_seconds)
            else :
                # 재시도 횟수 초과 에러 그대로 발생
                raise

In [15]:
# 후보군 통합  및 rerank 처리
from tqdm import tqdm

rerank_results = {}

for idx,row in tqdm(queries_df.iterrows()):
    qid = row['query_id']
    query_text = row['query_text']

    concat_candidates = bm25_candidates[qid] + dense_candidates[qid]

    # 순서 유지하며 중복 제거
    candidates = dict.fromkeys(concat_candidates)

    # 후보 문서 전체를 cohere rerank로 재정렬한 뒤 상위 5개만 사용
    rerank_results[qid] = cohere_rerank(query_text,list(candidates.keys()))[:5]

    # trial key 호출 제한 걸리지 않도록 잠시 대기
    time.sleep(6)

30it [03:08,  6.30s/it]


In [ ]:
rerank_results

## 성능 평가

In [16]:
import numpy as np

def parse_relevant(relevant_str):
    """다중 정답 및 등급을 처리하기 위한 헬퍼 함수"""
    pairs = relevant_str.split(";")
    rel_dict = {}
    for pair in pairs:
        doc_id, grade = pair.split("=")
        rel_dict[doc_id] = grade
    return rel_dict 

def compute_metrics(predicted, relevant_dict, k=5):
    relevant_docs = set(relevant_dict.keys())
    top_k = predicted[:k]
    hits = sum(1 for doc in top_k if doc in relevant_docs)
    precision = hits / k
    total_relevant = len(relevant_docs)
    recall = hits / total_relevant if total_relevant > 0 else 0 
    rr = 0
    for idx, doc in enumerate(top_k):
        if doc in relevant_docs:
            rr = 1 / (idx + 1)
            break
    num_correct = 0
    precision_sum = 0
    for i, doc in enumerate(top_k):
        if doc in relevant_docs:
            num_correct += 1
            precision_sum += num_correct / (i + 1)
    denominator = min(total_relevant, k)
    ap = precision_sum / denominator if denominator > 0 else 0
    return precision, recall, rr, ap

def evaluate_all(method_results, queries_df, k=5):
    prec_list, rec_list, rr_list, ap_list = [], [], [], []
    for idx, row in queries_df.iterrows():
        qid = row['query_id']
        relevant_dict = parse_relevant(row['relevant_doc_ids'])
        predicted = method_results[qid]
        p, r, rr, ap = compute_metrics(predicted, relevant_dict, k)
        prec_list.append(p)
        rec_list.append(r)
        rr_list.append(rr)
        ap_list.append(ap)
    return {
        'Precision@k' : np.mean(prec_list),
        'Recall@k' : np.mean(rec_list),
        'MRR' : np.mean(rr_list),
        'MAP' : np.mean(ap_list),
    }

In [17]:
bm25_results = {qid: lst[:5] for qid,lst in bm25_candidates.items()}
dense_results = {qid: lst[:5] for qid,lst in dense_candidates.items()}

bm25_metrics =  evaluate_all(bm25_results,queries_df)
dense_metrics = evaluate_all(dense_results,queries_df)
rerank_metrics = evaluate_all(rerank_results,queries_df)


metrics_df = pd.DataFrame({
    'Metric' : ["Precision@k","Recall@k","MRR","MAP"],
    'Dense' : [dense_metrics["Precision@k"],dense_metrics["Recall@k"],dense_metrics["MRR"],dense_metrics["MAP"]],
    'bm25' : [bm25_metrics["Precision@k"],bm25_metrics["Recall@k"],bm25_metrics["MRR"],bm25_metrics["MAP"]],
    'rerank' : [rerank_metrics["Precision@k"],rerank_metrics["Recall@k"],rerank_metrics["MRR"],rerank_metrics["MAP"]],
})
metrics_df

,Metric,Dense,bm25,rerank
0,Precision@k,0.233333,0.246667,0.246667
1,Recall@k,0.975000,1.000000,1.000000
2,MRR,1.000000,0.983333,0.983333
3,MAP,0.975000,0.977778,0.984444
